# 01 · Data Join and Cleaning

This notebook builds the one-row-per-order analytical dataset used by the clustering and prediction notebooks. The main goal is to combine the relational Olist tables without duplicating orders and to keep placement-stage and delivery-stage information clearly separated.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

orders = pd.read_csv(RAW_DIR / 'olist_orders_dataset.csv')
items = pd.read_csv(RAW_DIR / 'olist_order_items_dataset.csv')
payments = pd.read_csv(RAW_DIR / 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(RAW_DIR / 'olist_order_reviews_dataset.csv')
products = pd.read_csv(RAW_DIR / 'olist_products_dataset.csv')
sellers = pd.read_csv(RAW_DIR / 'olist_sellers_dataset.csv')
customers = pd.read_csv(RAW_DIR / 'olist_customers_dataset.csv')
translation = pd.read_csv(RAW_DIR / 'product_category_name_translation.csv')

## Cohort and target

The comparison is restricted to delivered orders so both prediction stages are evaluated on the same completed-order cohort. When an order has more than one review record, I keep the most recent one and define `review_bad = 1` for scores 1–2.

In [ ]:
order_date_cols = [
    'order_purchase_timestamp','order_approved_at',
    'order_delivered_carrier_date','order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for c in order_date_cols:
    orders[c] = pd.to_datetime(orders[c], errors='coerce')

for c in ['review_creation_date','review_answer_timestamp']:
    reviews[c] = pd.to_datetime(reviews[c], errors='coerce')

orders = orders.loc[orders['order_status'].eq('delivered')].copy()

reviews_clean = (
    reviews.sort_values(
        ['order_id','review_creation_date','review_answer_timestamp'],
        na_position='first'
    )
    .drop_duplicates('order_id', keep='last')
    [['order_id','review_score','review_creation_date','review_answer_timestamp']]
    .copy()
)
reviews_clean['review_bad'] = (reviews_clean['review_score'] <= 2).astype('int8')

## Aggregate one-to-many tables

Items and payments can have several rows per order, so they are summarized before the order-level merge. This prevents duplicated targets.

In [ ]:
products_clean = products.merge(
    translation,
    on='product_category_name',
    how='left'
)
products_clean['product_category_name_english'] = (
    products_clean['product_category_name_english'].fillna('unknown')
)
products_clean['product_volume_cm3'] = (
    products_clean['product_length_cm']
    * products_clean['product_height_cm']
    * products_clean['product_width_cm']
)

item_detail = items.merge(
    products_clean[[
        'product_id','product_category_name_english',
        'product_weight_g','product_volume_cm3'
    ]],
    on='product_id',
    how='left'
)

items_agg = (
    item_detail.groupby('order_id')
    .agg(
        n_items=('order_item_id','count'),
        n_distinct_sellers=('seller_id','nunique'),
        n_distinct_products=('product_id','nunique'),
        n_product_categories=('product_category_name_english','nunique'),
        total_price=('price','sum'),
        total_freight=('freight_value','sum'),
        avg_item_price=('price','mean'),
        avg_product_weight_g=('product_weight_g','mean'),
        max_product_weight_g=('product_weight_g','max'),
        avg_product_volume_cm3=('product_volume_cm3','mean'),
        max_product_volume_cm3=('product_volume_cm3','max')
    )
    .reset_index()
)

primary_item = (
    item_detail.sort_values(
        ['order_id','price','order_item_id'],
        ascending=[True,False,True]
    )
    .drop_duplicates('order_id')
    [['order_id','product_id','seller_id','product_category_name_english']]
    .rename(columns={
        'product_id':'primary_product_id',
        'seller_id':'primary_seller_id',
        'product_category_name_english':'primary_product_category'
    })
)
items_agg = items_agg.merge(primary_item, on='order_id', how='left')

payments_agg = (
    payments.groupby('order_id')
    .agg(
        total_payment_value=('payment_value','sum'),
        n_payment_installments=('payment_installments','max'),
        n_payment_methods=('payment_type','nunique')
    )
    .reset_index()
)

payment_by_type = (
    payments.groupby(['order_id','payment_type'], as_index=False)['payment_value']
    .sum()
)
primary_payment = (
    payment_by_type.sort_values(
        ['order_id','payment_value','payment_type'],
        ascending=[True,False,True]
    )
    .drop_duplicates('order_id')
    [['order_id','payment_type']]
    .rename(columns={'payment_type':'primary_payment_type'})
)
payments_agg = payments_agg.merge(primary_payment, on='order_id', how='left')

## Build the order-level master table

In [ ]:
customers_clean = customers[[
    'customer_id','customer_unique_id','customer_zip_code_prefix',
    'customer_city','customer_state'
]].copy()

sellers_clean = (
    sellers[[
        'seller_id','seller_zip_code_prefix','seller_city','seller_state'
    ]]
    .copy()
    .rename(columns={
        'seller_id':'primary_seller_id',
        'seller_zip_code_prefix':'primary_seller_zip_code_prefix',
        'seller_city':'primary_seller_city',
        'seller_state':'primary_seller_state'
    })
)

master = (
    orders
    .merge(customers_clean, on='customer_id', how='left')
    .merge(items_agg, on='order_id', how='inner')
    .merge(payments_agg, on='order_id', how='left')
    .merge(sellers_clean, on='primary_seller_id', how='left')
    .merge(reviews_clean, on='order_id', how='left')
)

assert master['order_id'].is_unique

## Feature engineering

Placement features use information available when the order is created. Delivery features are added only for the later model. Negative approval-to-carrier intervals are flagged as timestamp anomalies and set to missing.

In [ ]:
master['purchase_hour'] = master['order_purchase_timestamp'].dt.hour.astype('Int64')
master['purchase_day_of_week'] = master['order_purchase_timestamp'].dt.dayofweek.astype('Int64')
master['purchase_month'] = master['order_purchase_timestamp'].dt.month.astype('Int64')
master['is_weekend'] = master['purchase_day_of_week'].isin([5,6]).astype('int8')

master['promised_delivery_days'] = (
    master['order_estimated_delivery_date'] - master['order_purchase_timestamp']
).dt.total_seconds() / 86400

master['freight_to_price_ratio'] = np.where(
    master['total_price'] > 0,
    master['total_freight'] / master['total_price'],
    np.nan
)
master['avg_freight_per_item'] = np.where(
    master['n_items'] > 0,
    master['total_freight'] / master['n_items'],
    np.nan
)
master['same_customer_seller_state'] = (
    master['customer_state'].notna()
    & master['primary_seller_state'].notna()
    & master['customer_state'].eq(master['primary_seller_state'])
).astype('int8')

master['delivery_time_days'] = (
    master['order_delivered_customer_date'] - master['order_purchase_timestamp']
).dt.total_seconds() / 86400

master['delivery_vs_estimate_days'] = (
    master['order_delivered_customer_date'] - master['order_estimated_delivery_date']
).dt.total_seconds() / 86400

master['approval_delay_days'] = (
    master['order_approved_at'] - master['order_purchase_timestamp']
).dt.total_seconds() / 86400

master['carrier_pickup_delay_days_raw'] = (
    master['order_delivered_carrier_date'] - master['order_approved_at']
).dt.total_seconds() / 86400

master['carrier_timestamp_anomaly'] = (
    master['carrier_pickup_delay_days_raw'] < 0
).astype('int8')

master['carrier_pickup_delay_days'] = (
    master['carrier_pickup_delay_days_raw']
    .mask(master['carrier_pickup_delay_days_raw'] < 0, np.nan)
)

master['shipping_transit_days'] = (
    master['order_delivered_customer_date']
    - master['order_delivered_carrier_date']
).dt.total_seconds() / 86400

master['delivered_late'] = (
    master['delivery_vs_estimate_days'] > 0
).astype('int8')
master['late_days'] = master['delivery_vs_estimate_days'].clip(lower=0)
master['early_days'] = (-master['delivery_vs_estimate_days']).clip(lower=0)

## Final cohort and frozen split

Orders without an observed review or required delivery timestamps are removed. Categorical missing values are grouped as `unknown`; numeric missing values are left for the model-specific preprocessing. The train/test split is stratified and frozen with `random_state=42`.

In [ ]:
master = master.dropna(subset=['review_bad']).copy()
master = master.dropna(subset=[
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'delivery_time_days',
    'delivery_vs_estimate_days'
]).copy()

for c in [
    'customer_state','customer_city',
    'primary_seller_state','primary_seller_city',
    'primary_payment_type','primary_product_category'
]:
    master[c] = master[c].fillna('unknown')

train_ids, test_ids = train_test_split(
    master['order_id'],
    test_size=0.20,
    random_state=42,
    stratify=master['review_bad']
)

split_map = pd.DataFrame({
    'order_id': list(train_ids) + list(test_ids),
    'split': ['train'] * len(train_ids) + ['test'] * len(test_ids)
})
master = master.merge(split_map, on='order_id', how='left')

assert len(master) == 95824
assert master['order_id'].is_unique
assert master['split'].value_counts().to_dict() == {
    'train': 76659,
    'test': 19165
}

## Save processed outputs

The processed master table is the single input used by the downstream clustering, modeling, interpretation and Power BI export steps.

In [ ]:
audit = pd.DataFrame({
    'check': [
        'Final modeling rows',
        'Unique order_id',
        'Negative-review rows',
        'Negative-review rate',
        'Train rows',
        'Test rows',
        'Carrier timestamp anomalies'
    ],
    'value': [
        len(master),
        master['order_id'].nunique(),
        int(master['review_bad'].sum()),
        float(master['review_bad'].mean()),
        int((master['split'] == 'train').sum()),
        int((master['split'] == 'test').sum()),
        int(master['carrier_timestamp_anomaly'].sum())
    ]
})

master.to_csv(PROCESSED_DIR / 'master_orders_clean_v2.csv', index=False)
master[['order_id','split','review_bad']].to_csv(
    PROCESSED_DIR / 'shared_train_test_split.csv',
    index=False
)
audit.to_csv(PROCESSED_DIR / 'data_quality_audit_v2.csv', index=False)

audit